# Hybrid v7 — blended-eval launch notebook

## 0. Setup

In [ ]:
import sys, json as _json, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists() and (p / "qpp").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qpp.hybrid import legs, sweep

RESULTS_DIR = Path(sweep.__file__).resolve().parent / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("results dir:", RESULTS_DIR)


## 1. Load the universe bundle

In [ ]:
bundle, AVAILABLE_LEGS, SKIPPED_LEGS = legs.build_universe_bundle()
print("available:", AVAILABLE_LEGS)
print("skipped:  ", SKIPPED_LEGS)

assert legs.REAL_LEG in AVAILABLE_LEGS, f"{legs.REAL_LEG!r} (D0/ObservedTrain) failed to load"
assert legs.OURS in AVAILABLE_LEGS, f"{legs.OURS!r} (D1/D8, the headline arms) failed to load"

T_EVAL_QIDS = legs.t_eval_qids(bundle)
OBSERVED_FULL = legs.leg_train_qids(bundle, legs.REAL_LEG)
GENERATED_QIDS_BY_LEG = {
    leg: legs.leg_train_qids(bundle, leg)
    for leg in [legs.OURS] + legs.BASELINE_LEGS
    if leg in AVAILABLE_LEGS
}

assert not (set(OBSERVED_FULL) & set(T_EVAL_QIDS)), "ObservedTrain overlaps T_eval!"
for leg, qids in GENERATED_QIDS_BY_LEG.items():
    assert not (set(qids) & set(T_EVAL_QIDS)), f"{leg} overlaps T_eval!"
print("\ncontamination check passed.")

## 2. Config

In [ ]:
DIVERSQL_ARMS = ["D1_hybrid_diversql"]

OTHER_GENERATOR_ARMS = [
    "D2_hybrid_sqlbarber", "D3_hybrid_bootstrapping_lcm", "D4_hybrid_sql_factory",
    "D5_hybrid_e2etune", "D6_hybrid_sqlstorm",
]

MODELS = sweep.ALL_MODELS  # ["random_forest", "xgboost", "lightgbm", "mlp", "gp_rbf", "mscn"]
CHECKPOINTS = sweep.DEFAULT_CHECKPOINTS  # [5, 10, 20, 50, 100, 200, 400, 700, 10**9]
N_SEEDS = sweep.DEFAULT_N_SEEDS  # 5
ARMS = DIVERSQL_ARMS + OTHER_GENERATOR_ARMS
COMPUTE_STATS = False  # per-checkpoint subset stats are not needed for the paper

SEED_BASE = legs.SEED
NUM_EPOCHS = sweep.DEFAULT_NUM_EPOCHS

print(f"models: {MODELS}")
print(f"checkpoints: {CHECKPOINTS}")
print(f"n_seeds: {N_SEEDS}")
print(f"arms: {ARMS}")
print(f"compute_stats: {COMPUTE_STATS}")
print(f"\nsweep scope: {len(ARMS)} arms x {N_SEEDS} seeds x up to {len(CHECKPOINTS)} checkpoints "
      f"x {len(MODELS)} models")
print(f"\ncost note: trimmed from sweep.ARMS's full 9 to {len(ARMS)} -- roughly "
      f"{3 * len(CHECKPOINTS) + 1}/{8 * len(CHECKPOINTS) + 1} of v4's per-seed checkpoint-units "
      "(a rough proxy only -- mscn's per-checkpoint retrain cost doesn't scale the same way "
      "plan-encoder fitting does, so time the smoke test rather than trust this ratio exactly).")

## 3. Run the sweep

`checkpoint_csv_path` makes this crash-safe -- every checkpoint's rows (now one per test_leg, plus
one `__all_blended__` row, instead of v4's single pre-aggregated row) are appended to disk as soon
as they're computed.

In [ ]:
_stamp = f"{dt.datetime.utcnow():%Y%m%dT%H%M%SZ}"
p_results = RESULTS_DIR / f"hybrid_results_{_stamp}.csv"
p_config = RESULTS_DIR / f"hybrid_config_{_stamp}.json"
print("results will be checkpointed incrementally to:", p_results.name)

_t0 = dt.datetime.utcnow()

df_results = sweep.run_sweep(
    bundle=bundle,
    observed_leg_full_qids=OBSERVED_FULL,
    generated_qids_by_leg=GENERATED_QIDS_BY_LEG,
    checkpoints=CHECKPOINTS,
    n_seeds=N_SEEDS,
    models=MODELS,
    arms=ARMS,
    seed_base=SEED_BASE,
    num_epochs=NUM_EPOCHS,
    compute_stats=COMPUTE_STATS,
    checkpoint_csv_path=p_results,
)

_elapsed = (dt.datetime.utcnow() - _t0).total_seconds()
print(f"\ndone in {_elapsed:.0f}s -- {len(df_results)} rows (more rows than v4 for the same sweep shape: "
      "one per test_leg per model per checkpoint now, not one pre-aggregated row)")
df_results.head(15)

## 4. Save results

In [ ]:
df_results.to_csv(p_results, index=False)
p_config.write_text(_json.dumps({
    "stamp": _stamp, "quick_smoke_test": "False", "models": MODELS,
    "checkpoints": CHECKPOINTS, "n_seeds": N_SEEDS, "seed_base": SEED_BASE, "arms": ARMS,
    "num_epochs": NUM_EPOCHS, "generated_test_frac": legs.GENERATED_TEST_FRAC,
    "n_t_eval": len(T_EVAL_QIDS), "n_observed_full": len(OBSERVED_FULL),
    "available_legs": AVAILABLE_LEGS, "skipped_legs": SKIPPED_LEGS,
    "elapsed_seconds": _elapsed,
}, indent=2))

print("saved:", p_results.name)
print("saved:", p_config.name)

## 5. Sanity-check the fix itself

Before trusting any downstream number: confirm `in_train_composition` matches
`sweep._ARM_TRAIN_LEGS` exactly, and that the `__all_blended__` sentinel row's metric really is the
pooled-over-everything number (no exclusion) -- i.e. what v4 would have reported for the identical
pool. If either check fails, nothing past this cell is trustworthy.

In [ ]:
if len(df_results):
    per_leg = df_results[df_results["test_leg"] != sweep.ALL_BLENDED_SENTINEL]
    for arm in per_leg["arm"].unique():
        expected = sweep._ARM_TRAIN_LEGS[arm]
        sub = per_leg[per_leg["arm"] == arm]
        mismatch = sub[sub["in_train_composition"] != sub["test_leg"].isin(expected)]
        assert mismatch.empty, f"{arm}: in_train_composition disagrees with _ARM_TRAIN_LEGS for {mismatch['test_leg'].unique()}"
    print("in_train_composition check passed: matches sweep._ARM_TRAIN_LEGS for every arm.")

    blended = df_results[df_results["test_leg"] == sweep.ALL_BLENDED_SENTINEL]
    per_leg_n_sum = per_leg.groupby(["arm", "n_observed", "reveal_seed", "model"])["n_test"].sum()
    blended_n = blended.set_index(["arm", "n_observed", "reveal_seed", "model"])["n_test"]
    assert (per_leg_n_sum.reindex(blended_n.index) == blended_n).all(), (
        "__all_blended__ row's n_test doesn't equal the sum of its per-test-leg rows -- "
        "some prediction went unaccounted for."
    )
    print("__all_blended__ row-count check passed: sums to exactly the per-test-leg breakdown.")
else:
    print("no rows yet -- run the sweep first.")

## 7. Off-diagonal vs. blended-all -- does the exclusion matter?

The comparison this whole notebook exists to make: `sweep.off_diagonal_metric` (the corrected
number, each arm's own training legs excluded) against `sweep.blended_all_metric` (v4's original
pooled number, recovered from the `__all_blended__` rows in this same run). If `D1`'s apparent
edge over `D0` shrinks once DiverSQL's own held-out slice is excluded from `D1`'s score, that's
the §0 asymmetry actually biting -- not just a theoretical concern.

In [ ]:
_metric = "log_mae"

if len(df_results):
    off_diag = sweep.off_diagonal_metric(df_results, metric=_metric)
    blended = sweep.blended_all_metric(df_results, metric=_metric)

    off_diag_mean = off_diag.groupby(["arm", "model"])[_metric].mean().rename("off_diagonal")
    blended_mean = blended.groupby(["arm", "model"])[_metric].mean().rename("blended_all")
    compare = pd.concat([off_diag_mean, blended_mean], axis=1)
    compare["off_diagonal_minus_blended_all"] = compare["off_diagonal"] - compare["blended_all"]
    display(compare.round(4))
    print("\npositive 'off_diagonal_minus_blended_all' = this arm looked BETTER under v4's pooled "
          "metric than it does once its own training legs are excluded -- i.e. v4 was crediting it "
          "partly for fitting its own training distribution.")
else:
    print("no rows yet -- run the sweep first.")

## 8. `D8` regression check, under both protocols

v4's own cheapest, highest-signal check (`../v4/hybrid-v4-blended-eval-plan.md` §3 item 3): does
`D8_generated_only` beat `D0_real_only`'s best checkpoint? Run under `off_diagonal_metric` (the
corrected version) and `blended_all_metric` (v4's original) side by side -- if they disagree,
that's the headline finding of this whole notebook, not a footnote.

In [ ]:
def _d8_check(df_metric: pd.DataFrame, label: str) -> pd.DataFrame:
    d8 = df_metric[df_metric["arm"] == "D8_generated_only"].groupby("model")[_metric].mean()
    d0_best = df_metric[df_metric["arm"] == "D0_real_only"].groupby("model")[_metric].min()
    check = pd.DataFrame({f"D8_generated_only ({label})": d8, f"D0_real_only_best ({label})": d0_best})
    check[f"D8_beats_D0_best ({label})"] = check[f"D8_generated_only ({label})"] < check[f"D0_real_only_best ({label})"]
    return check

if len(df_results):
    check_off_diag = _d8_check(sweep.off_diagonal_metric(df_results, metric=_metric), "off-diagonal")
    check_blended = _d8_check(sweep.blended_all_metric(df_results, metric=_metric), "blended-all")
    display(pd.concat([check_off_diag, check_blended], axis=1).round(4))

    n_off = check_off_diag.iloc[:, -1].sum()
    n_blended = check_blended.iloc[:, -1].sum()
    print(f"\noff-diagonal:  {n_off}/{len(check_off_diag)} models: D8 beats D0's best checkpoint.")
    print(f"blended-all:   {n_blended}/{len(check_blended)} models: D8 beats D0's best checkpoint.")
    if n_off != n_blended:
        print("\n*** the two protocols disagree on the D8-vs-D0 regression check -- resolve this "
              "before writing anything up. ***")
else:
    print("no rows yet -- run the sweep first.")

## 9. Supplementary: other-generator arms (representative + full checkpoint only)

The main sweep above (§3) scoped `ARMS` to `D0`/`D1`/`D1s`/`D8` only -- the "TPC-DS + DiverSQL
only" decision from §2 -- so `D2`..`D6` (SQLBarber, Bootstrapping-LCM, SQL-Factory, e2etune,
SQLStorm) never ran. If the goal now is a supervisor-facing comparison across every benchmark
generator, not just DiverSQL, this section adds those 5 arms back in -- but only at **two
checkpoints each**: a representative mid-scale point (`N=200`, comfortably past MLP's `N=20`
early-stopping floor) and the full point (`N=1012`), instead of the main sweep's full 9-point
grid. That keeps this affordable while still answering "how does each generator perform with a
realistic amount of real data, and at full scale."

**Cost caveat, read before running**: checkpoint count isn't the only cost driver. The plan
encoder (`fit_leg_plan_embeddings`) is fit once per `(arm, seed)` regardless of how many
checkpoints that arm has -- so this section still pays for `5 arms x 5 seeds = 25` encoder fits,
*more* than the main sweep's `4 arms x 5 seeds = 20`, even though it has far fewer total
checkpoints (`10` vs. the main sweep's `28`). The main sweep's actual measured cost was `87,812s`
(~24.4 hours) -- well over the ~3 hour estimate a naive checkpoint-count ratio gave in §2, because
that ratio undercounted the encoder-fit cost. This section's own cost is genuinely uncertain for
the same reason: don't assume "5/9 the arms, 2/9 the checkpoints" means "much less than 24.4
hours" without timing it. If you want a cheap sanity check first, temporarily set
`OTHER_GENERATOR_CHECKPOINTS = [200]` (drop the full point) and time one arm before committing to
all five.

Appends into the **same results CSV** the main sweep already wrote (resolved from the latest
config/CSV pair on disk, not assumed to still be in this kernel's memory -- safe to run in a fresh
kernel after just §0-§1) -- so §7/§8's `off_diagonal_metric`/`blended_all_metric` calls, and
everything in §10 below, see all 9 arms from one file with no merge step. Requires `bundle`/
`OBSERVED_FULL`/`GENERATED_QIDS_BY_LEG` from §1 already in memory.

In [ ]:
OTHER_GENERATOR_ARMS = [
    "D2_hybrid_sqlbarber", "D3_hybrid_bootstrapping_lcm", "D4_hybrid_sql_factory",
    "D5_hybrid_e2etune", "D6_hybrid_sqlstorm",
]
# 200: representative, past MLP's N=20 early-stopping floor. 10**9: clips to each leg's true full
# ObservedTrain size (1012) -- same sentinel convention as sweep.DEFAULT_CHECKPOINTS.
OTHER_GENERATOR_CHECKPOINTS = [200, 10**9]

# Same models/seeds/epochs as the main sweep -- only ARMS and CHECKPOINTS differ here.
_OTHER_MODELS = sweep.ALL_MODELS
_OTHER_N_SEEDS = sweep.DEFAULT_N_SEEDS
_OTHER_SEED_BASE = legs.SEED
_OTHER_NUM_EPOCHS = sweep.DEFAULT_NUM_EPOCHS
_OTHER_COMPUTE_STATS = False

# Resolve the main sweep's stamp by exact pattern (sweep.load_results) -- a bare
# hybrid_config_*.json glob also matches suffixed files and can pick the wrong one.
_, _main_config, _STAMP = sweep.load_results(RESULTS_DIR, drop_degenerate=False)
p_results = RESULTS_DIR / f"hybrid_results_{_STAMP}.csv"
print(f"appending into existing run {_STAMP} (currently has arms: {_main_config['arms']})")

_t0_other = dt.datetime.utcnow()

df_other_generators = sweep.run_sweep(
    bundle=bundle,
    observed_leg_full_qids=OBSERVED_FULL,
    generated_qids_by_leg=GENERATED_QIDS_BY_LEG,
    checkpoints=OTHER_GENERATOR_CHECKPOINTS,
    n_seeds=_OTHER_N_SEEDS,
    models=_OTHER_MODELS,
    arms=OTHER_GENERATOR_ARMS,
    seed_base=_OTHER_SEED_BASE,
    num_epochs=_OTHER_NUM_EPOCHS,
    compute_stats=_OTHER_COMPUTE_STATS,
    checkpoint_csv_path=p_results,
)

_elapsed_other = (dt.datetime.utcnow() - _t0_other).total_seconds()
print(f"\ndone in {_elapsed_other:.0f}s -- {len(df_other_generators)} new rows appended")

# Named OUTSIDE the hybrid_config_* pattern on purpose, so sweep.load_results never mistakes it
# for the main config.
p_other_config = RESULTS_DIR / f"hybrid_supplement_other_generators_{_STAMP}.json"
p_other_config.write_text(_json.dumps({
    "stamp": _STAMP, "supplementary_arms": OTHER_GENERATOR_ARMS,
    "supplementary_checkpoints": OTHER_GENERATOR_CHECKPOINTS,
    "n_seeds": _OTHER_N_SEEDS, "seed_base": _OTHER_SEED_BASE,
    "elapsed_seconds": _elapsed_other,
}, indent=2))
print("saved:", p_other_config.name)

## 10. Per-generator comparison at N=200 and full scale

Reload the full results file (all 9 arms now) and answer the actual question §9 was for: is
DiverSQL (`D1`) really the best hybrid partner, or do the other generators do as well or better?
This is the success criterion v4's design doc originally specified (`hybrid-v4-blended-eval-plan.md`
§3 item 2) that got dropped when §2 trimmed `ARMS` down to DiverSQL-only -- answered here under
v7's corrected off-diagonal scoring, at both checkpoints §9 ran.

In [ ]:
df_all, _, _ = sweep.load_results(RESULTS_DIR)  # dedupes re-appended checkpoints, drops degenerate GP fits
print(f"{len(df_all)} total rows across arms: {sorted(df_all.arm.unique())}")

_hybrid_arms = ["D1_hybrid_diversql"] + OTHER_GENERATOR_ARMS
_checkpoints_to_show = OTHER_GENERATOR_CHECKPOINTS[:-1] + [len(OBSERVED_FULL)]  # [200, 1012] as run

# Each generator vs D0 on the legs NEITHER trained on (sweep.common_leg_metric) -- per-arm
# off_diagonal_metric scores D_k and D0 on different leg sets, which biases D_k - D0.
for n in _checkpoints_to_show:
    cols = {}
    for arm in _hybrid_arms:
        c = sweep.common_leg_metric(df_all[df_all.n_observed == n], ["D0_real_only", arm], metric="log_mae")
        p = c.pivot_table(index=["model", "reveal_seed"], columns="arm", values="log_mae").dropna()
        cols[arm] = ((p["D0_real_only"] - p[arm]) / p["D0_real_only"] * 100).groupby("model").mean()
    pivoted = pd.DataFrame(cols)
    print(f"\n=== N={n}: % log-MAE improvement vs D0_real_only, common held-out legs ===")
    display(pivoted.round(2))
    print("best hybrid partner per model:")
    print(pivoted.idxmax(axis=1).to_string())

# Point df_results at the combined (all 9 arms) data so re-running §5/§7/§8 covers every arm.
df_results = df_all
print(f"\ndf_results now points at all {len(df_results.arm.unique())} arms -- "
      "re-run §5/§7/§8 above if you want those checks/comparisons over the full set.")

## Notes

- Training-pool machinery (arms, pools, blended `T_eval`) is byte-for-byte identical to `../v4/` --
  see `legs.py`/`pools.py`'s own docstrings.
- The fix is entirely in scoring: `sweep.py`'s `_score_rows` now writes one row per
  (arm, n_observed, reveal_seed, model, test_leg) instead of one pre-aggregated row, so
  `off_diagonal_metric` can exclude each arm's own training legs and `blended_all_metric` can still
  recover v4's original number from the same run.
- Any future change to the exclusion rule (different weighting, a different `generated_test_frac`,
  ...) is a filter over the saved CSV, not another rerun -- see `hybrid-v7-plan.md` §1.2, §3.
- `D7` (no-feedback ablation) is excluded from `ARMS` above until
  `gpt_generated_1000_tpcds_tokens_no_feedback` has an execution run (unchanged gap from v3/v4).
- The main sweep (§3) completed with `elapsed_seconds: 87811.8` (~24.4 hours) for
  `D0`/`D1`/`D1s`/`D8` at the full 9-checkpoint grid -- well over the ~3 hour estimate §2 gave,
  because that estimate's checkpoint-count ratio didn't account for the plan encoder's fixed
  per-(arm, seed) fitting cost. Keep that in mind when estimating §9's cost too.
- §9/§10 add the other 5 generators back in at two checkpoints each (`N=200`, full) rather than
  the full grid, and answer the "is DiverSQL the best hybrid partner" question v4's design doc
  originally asked for (`hybrid-v4-blended-eval-plan.md` §3 item 2) but §2's DiverSQL-only scoping
  had dropped.
- See `hybrid-v7-plan.md` §4 for the success criteria this sweep is meant to answer, stated before
  the result was known.